# Eval Workbench

This notebook runs the default eval workflow, reloads the generated artifacts from `outputs/`, and renders summary tables for quick iteration after code changes.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / "app").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if not (repo_root / "app").exists():
    raise RuntimeError("Could not locate the repository root from the current working directory.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

from app.notebook_eval import (
    build_item_error_table,
    build_persona_error_table,
    load_eval_metrics,
    load_eval_records,
    run_eval_notebook,
    split_item_error_table,
)

repo_root


PosixPath('/home/mdel2424/dev/eRisk_Honours')

In [ ]:
# Matches the default README eval command.
personas = 2
seed = 42
eval_mode = "mixed_holdout"
prompt_version = "v1"
max_api_calls = 1000
trace_level = "off"
save_diagnostics = True
debug_outputs = True
run_eval_now = True

output_dir = repo_root / "outputs"

In [ ]:
if run_eval_now:
    run_summary = run_eval_notebook(
        persona_count=personas,
        seed=seed,
        eval_mode=eval_mode,
        prompt_version=prompt_version,
        save_diagnostics=save_diagnostics,
        max_api_calls=max_api_calls,
        trace_level=trace_level,
        debug_outputs=debug_outputs,
        output_dir=output_dir,
    )
    resolved_output_dir = Path(run_summary["output_dir"])
else:
    run_summary = None
    resolved_output_dir = Path(output_dir)

print(f"Artifacts ready in: {resolved_output_dir}")
run_summary if run_summary is not None else {"output_dir": str(resolved_output_dir)}

Running eval: mode=mixed_holdout, personas=3, prompt=v1, live_status=on
Stop policy: MIN_TURNS=20 | MAX_TURNS=40 | STOP_CONFIDENCE=0.40
Confidence model: CONF_SUPPORT_TAU=1.25 | CONF_DEPTH_WEIGHT=0.70 | CONF_COVERAGE_WEIGHT=0.30 | CONF_UP_ALPHA=0.55 | CONF_DECAY_STREAK_START=6 | CONF_DECAY_PER_TURN=0.002 | CONF_DECAY_MAX=0.01 | CONF_MAX_DROP_PER_TURN=0.01
Risk/extractor controls: RISK_SENTINEL_FLAG_THRESHOLD=0.45 | RISK_SENTINEL_SHORTCIRCUIT_THRESHOLD=1.1 | RISK_SENTINEL_ACTIVE_SHORTCIRCUIT=0 | EXTRACTOR_MIN_RECORDS_TARGET=1
[eval 1/3 persona=1] cycle=5 turn=3 stage=detector_graph conf=5.4% calls=6/1000

In [ ]:
if "resolved_output_dir" not in globals():
    resolved_output_dir = Path(output_dir)

metrics_payload = load_eval_metrics(resolved_output_dir)
records_df = load_eval_records(resolved_output_dir)
persona_error_df = build_persona_error_table(records_df)
item_error_df = build_item_error_table(records_df)
item_error_views = split_item_error_table(item_error_df)

print(f"Loaded {len(records_df)} evaluated personas from {resolved_output_dir}")
records_df[["persona_id", "split", "family", "bdi_true", "bdi_pred"]].head()


Loaded 5 evaluated personas from /home/mdel2424/dev/eRisk_Honours/outputs


,persona_id,split,family,bdi_true,bdi_pred
0,1,eval,risk_leaning,28,26
1,2,eval,somatic_evasive,36,31
2,3,eval,cognitive_ruminative,27,26
3,4,eval,cognitive_ruminative,5,12
4,5,eval,control_stressed,2,9


In [ ]:
primary_metrics = dict(metrics_payload.get("primary_metrics", {}))
summary_metrics_df = pd.DataFrame(
    [
        {"metric": "primary_eval_split", "value": metrics_payload.get("primary_eval_split", "")},
        {"metric": "item_f1_macro_at_1", "value": primary_metrics.get("item_f1_macro_at_1", metrics_payload.get("item_f1_macro_at_1", 0.0))},
        {"metric": "item_mae", "value": primary_metrics.get("item_mae", metrics_payload.get("item_mae", 0.0))},
        {"metric": "bdi_mae", "value": primary_metrics.get("bdi_mae", metrics_payload.get("bdi_mae", 0.0))},
        {"metric": "avg_turns_to_decision", "value": primary_metrics.get("avg_turns_to_decision", metrics_payload.get("avg_turns_to_decision", 0.0))},
        {"metric": "objective", "value": primary_metrics.get("objective", metrics_payload.get("objective", 0.0))},
    ]
)
summary_metrics_df


,metric,value
0,primary_eval_split,overall_labeled
1,item_f1_macro_at_1,0.6905
2,item_mae,0.7048
3,bdi_mae,4.4
4,avg_turns_to_decision,40.0
5,objective,0.5405


In [ ]:
family_summary_df = (
    persona_error_df.groupby("family", dropna=False)
    .agg(
        profiles=("persona_id", "count"),
        avg_bdi_true=("bdi_true", "mean"),
        avg_bdi_pred=("bdi_pred", "mean"),
        avg_bdi_abs_error=("bdi_abs_error", "mean"),
    )
    .reset_index()
    .sort_values(["avg_bdi_abs_error", "family"], ascending=[False, True])
)
family_summary_df.style.format(
    {
        "avg_bdi_true": "{:.2f}",
        "avg_bdi_pred": "{:.2f}",
        "avg_bdi_abs_error": "{:.2f}",
    }
)


,family,profiles,avg_bdi_true,avg_bdi_pred,avg_bdi_abs_error
1,control_stressed,1,2.00,9.00,7.00
3,somatic_evasive,1,36.00,31.00,5.00
0,cognitive_ruminative,2,16.00,19.00,4.00
2,risk_leaning,1,28.00,26.00,2.00


In [ ]:
persona_error_df.head(15).style.format(
    {
        "bdi_true": "{:.0f}",
        "bdi_pred": "{:.0f}",
        "bdi_error": "{:+.0f}",
        "bdi_abs_error": "{:.0f}",
    }
)


,persona_id,split,family,source,bdi_true,bdi_pred,bdi_error,bdi_abs_error
0,4,eval,cognitive_ruminative,synthetic,5,12,+7,7
1,5,eval,control_stressed,synthetic,2,9,+7,7
2,2,eval,somatic_evasive,synthetic,36,31,-5,5
3,1,eval,risk_leaning,synthetic,28,26,-2,2
4,3,eval,cognitive_ruminative,synthetic,27,26,-1,1


## Item-Level Analysis

`mean_error = avg_predicted_item_score - avg_ground_truth_item_score`

- Negative values mean under-predicted items.
- Positive values mean over-predicted items.


In [ ]:
DISPLAY_COLUMNS = [
    "item_id",
    "symptom_name",
    "avg_pred",
    "avg_true",
    "mean_error",
    "abs_mean_error",
    "n_profiles",
]

def _hex_to_rgb(value: str):
    value = value.lstrip("#")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))

def _rgb_to_hex(rgb):
    return "#%02x%02x%02x" % tuple(max(0, min(255, int(channel))) for channel in rgb)

def _blend_colors(start_hex: str, end_hex: str, weight: float):
    start_rgb = _hex_to_rgb(start_hex)
    end_rgb = _hex_to_rgb(end_hex)
    weight = max(0.0, min(1.0, float(weight)))
    blended = [start + ((end - start) * weight) for start, end in zip(start_rgb, end_rgb)]
    return _rgb_to_hex(blended)

def _mean_error_style(value: float, scale: float) -> str:
    if pd.isna(value):
        return ""
    magnitude = min(abs(float(value)) / max(scale, 0.001), 1.0)
    if float(value) < 0:
        color = _blend_colors("#ffffff", "#d73027", magnitude)
    elif float(value) > 0:
        color = _blend_colors("#ffffff", "#4575b4", magnitude)
    else:
        color = "#ffffff"
    return f"background-color: {color};"

def style_item_error_table(frame: pd.DataFrame):
    if frame.empty:
        return frame.reindex(columns=DISPLAY_COLUMNS)
    scale = max(abs(float(frame["mean_error"].min())), abs(float(frame["mean_error"].max())), 0.001)
    return (
        frame[DISPLAY_COLUMNS]
        .style
        .format(
            {
                "avg_pred": "{:.3f}",
                "avg_true": "{:.3f}",
                "mean_error": "{:+.3f}",
                "abs_mean_error": "{:.3f}",
            }
        )
        .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])
    )


In [ ]:
style_item_error_table(item_error_views["all_items"])


/tmp/ipykernel_29405/1530208316.py:52: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,12,Loss of Interest,0.600,1.200,-0.600,0.600,5
1,15,Loss of Energy,1.000,1.600,-0.600,0.600,5
2,2,Pessimism,1.000,1.400,-0.400,0.400,5
3,4,Loss of Pleasure,1.000,1.400,-0.400,0.400,5
4,5,Guilty Feelings,0.800,1.200,-0.400,0.400,5
5,3,Past Failure,1.000,1.200,-0.200,0.200,5
6,8,Self-Criticalness,1.200,1.400,-0.200,0.200,5
7,14,Worthlessness,0.800,1.000,-0.200,0.200,5
8,13,Indecisiveness,1.000,1.000,+0.000,0.000,5
9,19,Concentration Difficulty,1.600,1.600,+0.000,0.000,5


In [ ]:
style_item_error_table(item_error_views["under_predicted"])


/tmp/ipykernel_29405/1530208316.py:52: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,12,Loss of Interest,0.600,1.200,-0.600,0.600,5
1,15,Loss of Energy,1.000,1.600,-0.600,0.600,5
2,2,Pessimism,1.000,1.400,-0.400,0.400,5
3,4,Loss of Pleasure,1.000,1.400,-0.400,0.400,5
4,5,Guilty Feelings,0.800,1.200,-0.400,0.400,5
5,3,Past Failure,1.000,1.200,-0.200,0.200,5
6,8,Self-Criticalness,1.200,1.400,-0.200,0.200,5
7,14,Worthlessness,0.800,1.000,-0.200,0.200,5


In [ ]:
style_item_error_table(item_error_views["over_predicted"])


/tmp/ipykernel_29405/1530208316.py:52: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,9,Suicidal Thoughts or Wishes,2.000,0.600,+1.400,1.400,5
1,6,Punishment Feelings,0.800,0.000,+0.800,0.800,5
2,21,Loss of Interest in Sex,0.600,0.200,+0.400,0.400,5
3,1,Sadness,0.800,0.600,+0.200,0.200,5
4,7,Self-Dislike,0.800,0.600,+0.200,0.200,5
5,10,Crying,0.800,0.600,+0.200,0.200,5
6,11,Agitation,1.000,0.800,+0.200,0.200,5
7,16,Changes in Sleeping Pattern,1.400,1.200,+0.200,0.200,5
8,17,Irritability,0.600,0.400,+0.200,0.200,5
9,18,Changes in Appetite,1.000,0.800,+0.200,0.200,5
